# 🥈 Institutional Order Flow & Intraday Analytics Dashboard

Welcome to the **MDK Trading Oracle** Interactive Silver Exploration & Trader Analytics Suite.

### 🎯 What This Interactive Dashboard Provides:
1. **Institutional Trader / Broker Deep Dive**: Select any broker (Bank of America `MLB`, İş Yatırım `IYM`, Yapı Kredi `YKR`, Ak Yatırım `AKM`, Garanti `GRM`, Ziraat `ZRY`, etc.) to inspect daily market share, net flow trajectories, sector allocations, and top accumulated vs distributed stocks.
2. **Stock-Level Institutional Execution & VWAP Footprint**: Select any BIST equity to overlay daily price candlesticks with institutional **Buy/Sell VWAP execution price levels** and **Top 5 Domestic Desk Divergences**.
3. **Parameterized Intraday Time-Window Progression**: Slices any trading day into **4 customizable time windows** (`Day Start` $\rightarrow$ `Morning` $\rightarrow$ `Lunch` $\rightarrow$ `Closing Session`) to observe how big players build, expand, or liquidate positions.
4. **Sector Rotation Heatmaps**: Cross-sectional institutional sector allocations across all 21 trading days in March 2026.

## 1. Connect to DuckDB & Initialize Environment

In [ ]:
import sys
from pathlib import Path

import duckdb
import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from mdk_trading_oracle.core.config import get_settings

# Connect with read_only=True for lock-free concurrency
settings = get_settings()
db_path = settings.database_path
conn = duckdb.connect(str(db_path), read_only=True)

print(f"✅ Connected to DuckDB (read-only): {db_path.resolve()}")

## 2. Fact Tables Overview & Row Counts

In [ ]:
silver_counts = conn.execute("""
    SELECT 'silver_daily_broker_summary (Stock x Broker)' AS table_name, COUNT(*) AS total_rows FROM silver_daily_broker_summary
    UNION ALL
    SELECT 'silver_daily_broker_overview (Broker Macro & Ranks)' AS table_name, COUNT(*) AS total_rows FROM silver_daily_broker_overview
    UNION ALL
    SELECT 'silver_daily_stock_summary (OHLCV, VWAP, CR5)' AS table_name, COUNT(*) AS total_rows FROM silver_daily_stock_summary
    UNION ALL
    SELECT 'silver_daily_sector_summary (Sector Inflows)' AS table_name, COUNT(*) AS total_rows FROM silver_daily_sector_summary
    UNION ALL
    SELECT 'silver_intraday_broker_window_summary (4 Windows)' AS table_name, COUNT(*) AS total_rows FROM silver_intraday_broker_window_summary
    UNION ALL
    SELECT 'silver_intraday_sector_window_summary (Sector Windows)' AS table_name, COUNT(*) AS total_rows FROM silver_intraday_sector_window_summary;
""").pl()

silver_counts

--- 
## 3. 🏦 Interactive Trader / Broker Deep Dive Dashboard
Select any brokerage house below to dynamically visualize their **market share trajectory**, **daily net flow (TL)**, **sector concentration**, and **top equities accumulated vs distributed**.

In [ ]:
# Fetch all available brokers sorted by primary target & total turnover
broker_list = conn.execute("""
    SELECT 
        broker_id,
        broker_name || ' (' || broker_id || ')' AS label,
        SUM(total_turnover_tl) AS total_turnover
    FROM silver_daily_broker_overview
    GROUP BY broker_id, broker_name
    ORDER BY total_turnover DESC;
""").fetchall()

broker_options = [(b[1], b[0]) for b in broker_list]
default_broker = 'MLB' if 'MLB' in [b[0] for b in broker_list] else broker_options[0][1]

broker_dropdown = widgets.Dropdown(
    options=broker_options,
    value=default_broker,
    description="Select Broker:",
    layout=widgets.Layout(width="450px")
)

def render_broker_dashboard(broker_id):
    # 1. Macro overview stats
    overview_df = conn.execute(f"""
        SELECT 
            trade_date,
            broker_id,
            broker_name,
            broker_category,
            is_primary_target,
            ROUND(total_buy_turnover_tl / 1e6, 2) AS buy_million_tl,
            ROUND(total_sell_turnover_tl / 1e6, 2) AS sell_million_tl,
            ROUND(net_flow_tl / 1e6, 2) AS net_flow_million_tl,
            ROUND(total_turnover_tl / 1e6, 2) AS total_turnover_million_tl,
            ROUND(market_turnover_share * 100, 2) AS market_share_pct,
            market_turnover_rank,
            market_net_flow_rank,
            is_top_5_broker,
            top_bought_symbol,
            top_sold_symbol,
            top_sector_name
        FROM silver_daily_broker_overview
        WHERE broker_id = '{broker_id}'
        ORDER BY trade_date ASC;
    """).pl()
    
    if overview_df.height == 0:
        print(f"No data found for broker {broker_id}")
        return
        
    broker_name = overview_df['broker_name'][0]
    tot_turnover = overview_df['total_turnover_million_tl'].sum() / 1000.0
    tot_net_flow = overview_df['net_flow_million_tl'].sum()
    avg_share = overview_df['market_share_pct'].mean()
    days_top5 = overview_df['is_top_5_broker'].sum()
    
    # Display KPI Cards
    net_flow_color = "#10ac84" if tot_net_flow > 0 else "#ee5253"
    display(HTML(f"""
        <div style='display: flex; gap: 15px; margin-bottom: 20px;'>
            <div style='background: #1e272e; padding: 15px 20px; border-radius: 8px; border-left: 4px solid #00d2d3;'>
                <div style='color: #8395a7; font-size: 12px;'>TOTAL TURNOVER</div>
                <div style='color: white; font-size: 20px; font-weight: bold;'>{tot_turnover:.2f} Billion TL</div>
            </div>
            <div style='background: #1e272e; padding: 15px 20px; border-radius: 8px; border-left: 4px solid {net_flow_color};'>
                <div style='color: #8395a7; font-size: 12px;'>MONTHLY NET FLOW</div>
                <div style='color: {net_flow_color}; font-size: 20px; font-weight: bold;'>{tot_net_flow:+.1f} Million TL</div>
            </div>
            <div style='background: #1e272e; padding: 15px 20px; border-radius: 8px; border-left: 4px solid #feca57;'>
                <div style='color: #8395a7; font-size: 12px;'>AVG MARKET SHARE</div>
                <div style='color: white; font-size: 20px; font-weight: bold;'>{avg_share:.2f}%</div>
            </div>
            <div style='background: #1e272e; padding: 15px 20px; border-radius: 8px; border-left: 4px solid #ff9ff3;'>
                <div style='color: #8395a7; font-size: 12px;'>DAYS IN TOP 5</div>
                <div style='color: white; font-size: 20px; font-weight: bold;'>{days_top5} / {overview_df.height} Days</div>
            </div>
        </div>
    """))
    
    # Dual-Axis Time Series Chart (Daily Net Flow vs Market Share)
    fig_ts = make_subplots(specs=[[{"secondary_y": True}]])
    
    colors = ["#10ac84" if x >= 0 else "#ee5253" for x in overview_df["net_flow_million_tl"].to_list()]
    
    fig_ts.add_trace(
        go.Bar(
            x=overview_df["trade_date"].to_list(),
            y=overview_df["net_flow_million_tl"].to_list(),
            name="Daily Net Flow (M TL)",
            marker_color=colors,
            opacity=0.8
        ),
        secondary_y=False
    )
    
    fig_ts.add_trace(
        go.Scatter(
            x=overview_df["trade_date"].to_list(),
            y=overview_df["market_share_pct"].to_list(),
            name="Market Share (%)",
            mode="lines+markers",
            line=dict(color="#00d2d3", width=3)
        ),
        secondary_y=True
    )
    
    fig_ts.update_layout(
        title=f"{broker_name} ({broker_id}) — Daily Net Flow & Market Turnover Share",
        xaxis_title="Date",
        template="plotly_dark",
        height=400,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    fig_ts.update_yaxes(title_text="Net Flow (Million TL)", secondary_y=False)
    fig_ts.update_yaxes(title_text="Market Share (%)", secondary_y=True)
    fig_ts.show()
    
    # 2. Sector Allocation & Top 10 Equities Divergence (Side by Side)
    sector_df = conn.execute(f"""
        SELECT 
            sector,
            ROUND(SUM(total_turnover_tl) / 1e6, 2) AS sector_turnover_million_tl,
            ROUND(SUM(net_flow_tl) / 1e6, 2) AS sector_net_flow_million_tl
        FROM silver_daily_sector_summary
        WHERE broker_id = '{broker_id}'
        GROUP BY sector
        ORDER BY sector_turnover_million_tl DESC;
    """).pl()
    
    top_stocks_df = conn.execute(f"""
        SELECT 
            symbol,
            ROUND(SUM(net_flow_tl) / 1e6, 2) AS net_flow_million_tl
        FROM silver_daily_broker_summary
        WHERE broker_id = '{broker_id}'
        GROUP BY symbol
        ORDER BY net_flow_million_tl DESC;
    """).pl()
    
    top_bought = top_stocks_df.head(6)
    top_sold = top_stocks_df.tail(6)
    diverging_stocks = pl.concat([top_bought, top_sold]).sort("net_flow_million_tl", descending=False)
    
    fig_stocks = px.bar(
        diverging_stocks.to_pandas(),
        x="net_flow_million_tl",
        y="symbol",
        orientation="h",
        color="net_flow_million_tl",
        color_continuous_scale="RdYlGn",
        title=f"{broker_id} — Top Accumulated vs Distributed Equities (Monthly Net TL)",
        labels={"net_flow_million_tl": "Net Flow (Million TL)", "symbol": "Stock Ticker"},
        template="plotly_dark"
    )
    fig_stocks.update_layout(height=400)
    fig_stocks.show()

widgets.interact(render_broker_dashboard, broker_id=broker_dropdown);

--- 
## 4. 📈 Interactive Stock Deep Dive: Price Candlesticks & Institutional VWAP Overlays
Select any stock ticker and broker to overlay the **daily price action (OHLCV)** with the **institution's actual execution VWAP** and view the **Top 5 Domestic Desk net flow divergence**.

In [ ]:
# Available symbols
symbols_list = conn.execute("""
    SELECT DISTINCT symbol, symbol_name 
    FROM silver_daily_stock_summary 
    ORDER BY symbol ASC;
""").fetchall()

symbol_options = [(f"{s[0]} — {s[1]}", s[0]) for s in symbols_list]

stock_dropdown = widgets.Dropdown(
    options=symbol_options,
    value='THYAO' if 'THYAO' in [s[0] for s in symbols_list] else symbol_options[0][1],
    description="Select Stock:",
    layout=widgets.Layout(width="400px")
)

overlay_broker_dropdown = widgets.Dropdown(
    options=broker_options,
    value='MLB',
    description="Overlay Broker:",
    layout=widgets.Layout(width="400px")
)

def render_stock_dashboard(symbol, broker_id):
    # 1. Fetch stock daily market summary
    stock_df = conn.execute(f"""
        SELECT 
            trade_date,
            symbol,
            symbol_name,
            sector,
            open_price,
            high_price,
            low_price,
            close_price,
            market_vwap,
            daily_return_pct * 100 AS daily_return_pct,
            total_volume,
            ROUND(total_turnover_tl / 1e6, 2) AS total_turnover_million_tl,
            top_buyer_broker_id,
            top_seller_broker_id,
            ROUND(top_5_concentration_ratio * 100, 1) AS cr5_pct,
            ROUND(top_5_domestic_net_flow_tl / 1e6, 2) AS top5_domestic_net_flow_million_tl
        FROM silver_daily_stock_summary
        WHERE symbol = '{symbol}'
        ORDER BY trade_date ASC;
    """).pl()
    
    # 2. Fetch selected broker's specific execution on this stock
    broker_stock_df = conn.execute(f"""
        SELECT 
            trade_date,
            buy_vwap,
            sell_vwap,
            total_vwap,
            ROUND(net_flow_tl / 1e6, 2) AS broker_net_flow_million_tl,
            ROUND(broker_symbol_turnover_share * 100, 2) AS broker_share_pct
        FROM silver_daily_broker_summary
        WHERE symbol = '{symbol}' AND broker_id = '{broker_id}'
        ORDER BY trade_date ASC;
    """).pl()
    
    merged_df = stock_df.join(broker_stock_df, on="trade_date", how="left").fill_null(0.0)
    
    # Create Dual-Pane Plot (Upper: Candlestick + VWAPs, Lower: Net Flow Divergence)
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.65, 0.35],
        subplot_titles=[f"{symbol} Daily Candlesticks & {broker_id} Execution VWAP", f"{broker_id} Net Flow vs Domestic Top-5 Desks (Million TL)"]
    )
    
    # Candlestick
    fig.add_trace(
        go.Candlestick(
            x=merged_df["trade_date"].to_list(),
            open=merged_df["open_price"].to_list(),
            high=merged_df["high_price"].to_list(),
            low=merged_df["low_price"].to_list(),
            close=merged_df["close_price"].to_list(),
            name=f"{symbol} Price"
        ),
        row=1, col=1
    )
    
    # Market VWAP
    fig.add_trace(
        go.Scatter(
            x=merged_df["trade_date"].to_list(),
            y=merged_df["market_vwap"].to_list(),
            mode="lines",
            name="Market VWAP",
            line=dict(color="#dfe4ea", width=1.5, dash="dot")
        ),
        row=1, col=1
    )
    
    # Broker Buy VWAP
    fig.add_trace(
        go.Scatter(
            x=merged_df["trade_date"].to_list(),
            y=[y if y > 0 else None for y in merged_df["buy_vwap"].to_list()],
            mode="lines+markers",
            name=f"{broker_id} Buy VWAP",
            line=dict(color="#2ed573", width=2, dash="dash"),
            marker=dict(size=6, symbol="triangle-up")
        ),
        row=1, col=1
    )
    
    # Broker Sell VWAP
    fig.add_trace(
        go.Scatter(
            x=merged_df["trade_date"].to_list(),
            y=[y if y > 0 else None for y in merged_df["sell_vwap"].to_list()],
            mode="lines+markers",
            name=f"{broker_id} Sell VWAP",
            line=dict(color="#ff4757", width=2, dash="dash"),
            marker=dict(size=6, symbol="triangle-down")
        ),
        row=1, col=1
    )
    
    # Lower Subplot: Broker Net Flow
    fig.add_trace(
        go.Bar(
            x=merged_df["trade_date"].to_list(),
            y=merged_df["broker_net_flow_million_tl"].to_list(),
            name=f"{broker_id} Net Flow",
            marker_color=["#2ed573" if x >= 0 else "#ff4757" for x in merged_df["broker_net_flow_million_tl"].to_list()],
            opacity=0.85
        ),
        row=2, col=1
    )
    
    # Lower Subplot: Top-5 Domestic Desks Divergence Line
    fig.add_trace(
        go.Scatter(
            x=merged_df["trade_date"].to_list(),
            y=merged_df["top5_domestic_net_flow_million_tl"].to_list(),
            name="Top-5 Domestic Net Flow",
            line=dict(color="#eccc68", width=2)
        ),
        row=2, col=1
    )
    
    fig.update_layout(
        template="plotly_dark",
        height=650,
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    fig.show()

widgets.interact(render_stock_dashboard, symbol=stock_dropdown, broker_id=overlay_broker_dropdown);

--- 
## 5. ⏱ Interactive Intraday 4-Window Trajectory Explorer
Inspect the **4 parameterized intraday time windows** (`day_start`, `morning_to_lunch`, `lunch_to_15`, `closing_session`) for any trading day to track how order flows evolve from the Opening Call into the Close.

In [ ]:
# Available Dates
dates_list = conn.execute("SELECT DISTINCT trade_date FROM silver_daily_stock_summary ORDER BY trade_date DESC;").fetchall()
date_options = [str(d[0]) for d in dates_list]

date_dropdown = widgets.Dropdown(
    options=date_options,
    value=date_options[0],
    description="Trade Date:",
    layout=widgets.Layout(width="350px")
)

intraday_symbol_dropdown = widgets.Dropdown(
    options=symbol_options,
    value='THYAO',
    description="Stock:",
    layout=widgets.Layout(width="350px")
)

intraday_broker_dropdown = widgets.Dropdown(
    options=broker_options,
    value='MLB',
    description="Broker:",
    layout=widgets.Layout(width="350px")
)

def render_intraday_dashboard(trade_date, symbol, broker_id):
    intraday_df = conn.execute(f"""
        SELECT 
            window_name,
            window_order,
            window_start_time,
            ROUND(buy_volume, 0) AS buy_volume,
            ROUND(buy_turnover_tl / 1e6, 2) AS buy_million_tl,
            ROUND(buy_vwap, 2) AS buy_vwap,
            ROUND(sell_volume, 0) AS sell_volume,
            ROUND(sell_turnover_tl / 1e6, 2) AS sell_million_tl,
            ROUND(sell_vwap, 2) AS sell_vwap,
            ROUND(net_volume, 0) AS net_volume,
            ROUND(net_flow_tl / 1e6, 2) AS net_flow_million_tl,
            trade_count
        FROM silver_intraday_broker_window_summary
        WHERE trade_date = '{trade_date}' AND symbol = '{symbol}' AND broker_id = '{broker_id}'
        ORDER BY window_order ASC;
    """).pl()
    
    if intraday_df.height == 0:
        print(f"No intraday trades recorded for {broker_id} on {symbol} for date {trade_date}.")
        return
        
    # Render Table
    print(f"📊 Intraday Windows Summary for {broker_id} on {symbol} ({trade_date}):")
    display(intraday_df.to_pandas())
    
    # Waterfall / Progression Chart
    fig_win = go.Figure()
    
    fig_win.add_trace(go.Bar(
        x=intraday_df["window_name"].to_list(),
        y=intraday_df["buy_million_tl"].to_list(),
        name="Buy Turnover (M TL)",
        marker_color="#2ed573"
    ))
    
    fig_win.add_trace(go.Bar(
        x=intraday_df["window_name"].to_list(),
        y=intraday_df["sell_million_tl"].to_list(),
        name="Sell Turnover (M TL)",
        marker_color="#ff4757"
    ))
    
    fig_win.add_trace(go.Scatter(
        x=intraday_df["window_name"].to_list(),
        y=intraday_df["net_flow_million_tl"].to_list(),
        mode="lines+markers+text",
        text=[f"{v:+.1f}M" for v in intraday_df["net_flow_million_tl"].to_list()],
        textposition="top center",
        name="Net Flow (M TL)",
        line=dict(color="#feca57", width=3)
    ))
    
    fig_win.update_layout(
        title=f"{broker_id} — Intraday 4-Window Execution Breakdown on {symbol} ({trade_date})",
        xaxis_title="Intraday Time Window",
        yaxis_title="Turnover (Million TL)",
        template="plotly_dark",
        height=450,
        barmode="group"
    )
    fig_win.show()

widgets.interact(
    render_intraday_dashboard,
    trade_date=date_dropdown,
    symbol=intraday_symbol_dropdown,
    broker_id=intraday_broker_dropdown
);

--- 
## 6. 🔄 Interactive Sector Rotation Heatmap Matrix
Select any institution below to generate their **cross-sectional sector net flow heatmap** across all trading days in March 2026.

In [ ]:
sector_broker_dropdown = widgets.Dropdown(
    options=broker_options,
    value='MLB',
    description="Select Broker:",
    layout=widgets.Layout(width="400px")
)

def render_sector_heatmap(broker_id):
    sector_flows = conn.execute(f"""
        SELECT 
            trade_date,
            sector,
            ROUND(SUM(net_flow_tl) / 1e6, 2) AS net_flow_million_tl
        FROM silver_daily_sector_summary
        WHERE broker_id = '{broker_id}'
        GROUP BY trade_date, sector
        ORDER BY trade_date ASC, sector ASC;
    """).pl()
    
    if sector_flows.height == 0:
        print(f"No sector activity found for {broker_id}")
        return
        
    pivot_df = sector_flows.to_pandas().pivot(index="sector", columns="trade_date", values="net_flow_million_tl").fillna(0)
    
    fig_h = px.imshow(
        pivot_df,
        labels=dict(x="Date", y="Sector", color="Net Flow (M TL)"),
        title=f"{broker_id} — Daily Sector Net Flow Allocation Matrix (Million TL)",
        color_continuous_scale="RdYlGn",
        aspect="auto",
        template="plotly_dark"
    )
    fig_h.update_layout(height=500)
    fig_h.show()

widgets.interact(render_sector_heatmap, broker_id=sector_broker_dropdown);

## 💡 Next Steps: Gold Layer Predictive Modeling & Decision Rules

With these interactive visual dashboards, you can now seamlessly inspect any market participant, stock, or sector.

### Ready for Modeling:
1. **Monday Open Weekly Directional Forecaster** (training model on Friday close BofA inventories vs Monday morning returns).
2. **Intraday Flow Expansion Model** (predicting closing window volume expansion given opening window accumulation).
3. **BofA vs Top-5 Domestic Desk Divergence Reversal Signal**.